In [ ]:
import sqlite3, json, math, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

DB_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'wave_pool_weather.db'))
REAL_MAX_F = 91.0
REAL_MAX_C = (REAL_MAX_F - 32) * 5 / 9

def c2f(c): return c * 9/5 + 32

print(f'DB: {DB_PATH}  exists={os.path.exists(DB_PATH)}')

In [ ]:
# ── Load all locations, filter corrupted rows (air_C >= 45 are F values stored as C)
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("SELECT id, name FROM locations ORDER BY name")
locations = {r['name']: r['id'] for r in cur.fetchall()}

def load_location(loc_id):
    cur.execute("""
        SELECT pt.date, pt.temp AS pool_C, pt.heat_fluxes,
               at.temp AS air_C, at.wind_speed, at.humidity, at.solar_radiation
        FROM pool_temps pt
        JOIN air_temps at ON pt.location_id=at.location_id AND pt.date=at.date
        WHERE pt.location_id=? AND CAST(at.temp AS REAL) < 45
        ORDER BY pt.date
    """, (loc_id,))
    return cur.fetchall()

all_data = {name: load_location(lid) for name, lid in locations.items()}
conn.close()

for name, rows in all_data.items():
    print(f'{name:35s}  {len(rows)} rows  {rows[0]["date"]} → {rows[-1]["date"]}')

## Plot 1 — Heat flux breakdown (Waco)
Stack plot showing how each flux component contributes to the daily ΔT.
The evaporation term is the one suspected of under-correcting.

In [ ]:
rows = all_data['Waco']
dates = [datetime.strptime(r['date'], '%Y-%m-%d') for r in rows]

def flux(rows, key):
    return np.array([json.loads(r['heat_fluxes'])[key] for r in rows])

q_solar  = flux(rows, 'Q_solar_Wm2')
q_lw     = flux(rows, 'Q_lw_net_Wm2')
q_conv   = flux(rows, 'Q_conv_Wm2')
q_evap   = flux(rows, 'Q_evap_Wm2')
q_ground = flux(rows, 'Q_ground_Wm2')
q_total  = flux(rows, 'Q_total_Wm2')
pool_F   = np.array([c2f(float(r['pool_C'])) for r in rows])
air_F    = np.array([c2f(float(r['air_C'])) for r in rows])

fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

# — Top: stacked fluxes
ax = axes[0]
pos = np.maximum(q_solar, 0)
neg = q_lw + q_conv + q_evap + q_ground
ax.bar(dates, q_solar,  width=0.8, label='Q_solar',  color='#f9a825', alpha=0.85)
ax.bar(dates, q_lw,     width=0.8, label='Q_lw_net', color='#1565c0', alpha=0.85)
ax.bar(dates, q_conv,   width=0.8, label='Q_conv',   color='#43a047', alpha=0.85, bottom=q_lw)
ax.bar(dates, q_evap,   width=0.8, label='Q_evap',   color='#e53935', alpha=0.85, bottom=q_lw+q_conv)
ax.bar(dates, q_ground, width=0.8, label='Q_ground', color='#6d4c41', alpha=0.85, bottom=q_lw+q_conv+q_evap)
ax.plot(dates, q_total, color='black', lw=2, label='Q_total')
ax.axhline(0, color='black', lw=0.7)
ax.set_ylabel('W m⁻²')
ax.set_title('Waco BSR — Heat Flux Components')
ax.legend(fontsize=9, loc='upper left', ncol=3)
ax.grid(True, alpha=0.25, axis='y')

# — Middle: evap as fraction of solar (the bias diagnostic)
ax2 = axes[1]
ratio = np.abs(q_evap) / np.maximum(q_solar, 1.0)
ax2.bar(dates, ratio, width=0.8, color='#e53935', alpha=0.75)
ax2.axhline(1.0, color='black', ls='--', lw=1.2, label='|Q_evap| = Q_solar')
ax2.axhline(ratio.mean(), color='navy', ls=':', lw=1.2, label=f'mean={ratio.mean():.3f}')
ax2.set_ylabel('|Q_evap| / Q_solar')
ax2.set_title('Evaporation / Solar ratio — should be ≥ 1.0 to offset solar gain')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.25, axis='y')

# — Bottom: pool vs air temp
ax3 = axes[2]
ax3.plot(dates, air_F,  color='#e07b39', lw=1.3, alpha=0.7, label='Air temp')
ax3.plot(dates, pool_F, color='#1565c0', lw=2.0, label='Pool temp (stored)')
ax3.axhline(REAL_MAX_F, color='red', ls='--', lw=1.3, label=f'{REAL_MAX_F}°F cap')
ax3.set_ylabel('°F')
ax3.set_title('Pool vs Air Temperature')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.25)

axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[2].xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'Mean |Q_evap|/Q_solar:        {ratio.mean():.3f}  (want ≥ 1.0)')
hot = air_F > 90
print(f'Mean |Q_evap|/Q_solar hot days:{ratio[hot].mean():.3f}  ({hot.sum()} hot days)')

## Plot 2 — Bias correction methods vs. baseline (Waco)
Inline thermal model stepped forward from clean seed, all four methods compared.

In [ ]:
# Inline thermal model — mirrors pull_api_data.py
SIGMA_SB=5.67e-8; EPSILON_WATER=0.97; ALBEDO_WATER=0.06
RHO_WATER=1000.0; CP_WATER=4186.0; L_VAP=2.45e6
U_BOTTOM = 1.0 / (0.3048/1.4 + 2.0/1.5)
DEPTH_M=2.0; GROUND_TEMP_C=20.0

def sat_vp(T_C):
    return 0.6108 * math.exp(17.27 * T_C / (T_C + 237.3))

def sky_eps(T_C, RH_pct):
    T_K=T_C+273.15; RH=max(0.0, min(1.0, RH_pct/100.0))
    e_hPa=sat_vp(T_C)*RH*10.0
    return min(1.0, 1.24*(e_hPa/T_K)**(1/7))

def penman_evap(T_pool_C, T_air_C, RH_pct, wind_ms, solar_Wm2):
    RH=max(0.0, min(1.0, RH_pct/100.0))
    e_s_air=sat_vp(T_air_C)
    delta=4098.0*e_s_air/(T_air_C+237.3)**2
    Rn=solar_Wm2*86400.0/1e6; lam=2.45; gamma=0.067
    f_u=6.43*(1.0+0.536*wind_ms)
    Ea=f_u*(sat_vp(T_pool_C)-e_s_air*RH)
    E_mm=max(0.0,(delta*(Rn/lam)+gamma*Ea)/(delta+gamma))
    return -(E_mm/1000.0/86400.0*L_VAP)

def step(T_pool_C, T_air_C, RH_pct, wind_ms, solar_MJm2, method='none',
         ce_mult=1.25, wind_floor=1.5):
    T_pool_K=T_pool_C+273.15; T_air_K=T_air_C+273.15
    RH=max(0.0, min(1.0, RH_pct/100.0))
    solar_Wm2=max(0.0, float(solar_MJm2 or 15.0))*1e6/86400.0
    Q_solar=(1.0-ALBEDO_WATER)*solar_Wm2
    eps_sky=sky_eps(T_air_C, RH_pct)
    Q_lw_net=eps_sky*SIGMA_SB*T_air_K**4-EPSILON_WATER*SIGMA_SB*T_pool_K**4
    h_c=5.7+3.8*wind_ms
    Q_conv=h_c*(T_air_C-T_pool_C)
    EVAP_BASE=1.2*1.3e-3*L_VAP*0.622/101.325
    e_s_pool=sat_vp(T_pool_C); e_a=sat_vp(T_air_C)*RH
    if method=='evap_multiplier':
        Q_evap=-(EVAP_BASE*ce_mult)*wind_ms*(e_s_pool-e_a)
    elif method=='wind_floor':
        Q_evap=-EVAP_BASE*max(wind_ms,wind_floor)*(e_s_pool-e_a)
    elif method=='penman':
        Q_evap=penman_evap(T_pool_C, T_air_C, RH_pct, wind_ms, solar_Wm2)
    else:
        Q_evap=-EVAP_BASE*wind_ms*(e_s_pool-e_a)
    Q_ground=-U_BOTTOM*(T_pool_C-GROUND_TEMP_C)
    Q_total=Q_solar+Q_lw_net+Q_conv+Q_evap+Q_ground
    dT=Q_total*86400.0/(RHO_WATER*DEPTH_M*CP_WATER)
    return T_pool_C+dT, Q_evap

print('Model ready.')

In [ ]:
rows = all_data['Waco']
seed = float(rows[0]['pool_C'])
T = {m: seed for m in ('none','evap_multiplier','wind_floor','penman')}

rec = []
for r in rows:
    air_C = float(r['air_C'])
    RH    = float(r['humidity'] or 50.0)
    wind  = float(r['wind_speed'] or 3.0)
    solar = float(r['solar_radiation']) if r['solar_radiation'] is not None else 15.0
    res   = {'date': datetime.strptime(r['date'], '%Y-%m-%d'), 'air_C': air_C,
             'stored': float(r['pool_C'])}
    for m in T:
        T[m], _ = step(T[m], air_C, RH, wind, solar, method=m)
        res[m] = T[m]
    rec.append(res)

dates_s  = np.array([r['date']              for r in rec])
air_F    = np.array([c2f(r['air_C'])        for r in rec])
stored_F = np.array([c2f(r['stored'])       for r in rec])
none_F   = np.array([c2f(r['none'])         for r in rec])
mult_F   = np.array([c2f(r['evap_multiplier']) for r in rec])
floor_F  = np.array([c2f(r['wind_floor'])   for r in rec])
pen_F    = np.array([c2f(r['penman'])        for r in rec])

print(f'none:            max={none_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(none_F>REAL_MAX_F).sum()}')
print(f'evap_mult ×1.25: max={mult_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(mult_F>REAL_MAX_F).sum()}')
print(f'wind_floor 1.5:  max={floor_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(floor_F>REAL_MAX_F).sum()}')
print(f'penman:          max={pen_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(pen_F>REAL_MAX_F).sum()}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)

# — Top: all methods time series
ax = axes[0]
ax.fill_between(dates_s, air_F, alpha=0.12, color='#e07b39')
ax.plot(dates_s, air_F,   color='#e07b39', lw=1.2, alpha=0.7,  label='Air temp')
ax.plot(dates_s, none_F,  color='#aaa',    lw=1.5, ls='--',     label='none (baseline)')
ax.plot(dates_s, mult_F,  color='#2196f3', lw=2.2,              label='evap_mult ×1.25')
ax.plot(dates_s, floor_F, color='#43a047', lw=1.8, ls='-.',     label='wind_floor 1.5 m/s')
ax.plot(dates_s, pen_F,   color='#ab47bc', lw=1.8, ls=':',      label='penman')
ax.axhline(REAL_MAX_F, color='red', lw=1.4, ls='--', label=f'{REAL_MAX_F}°F cap')
ax.set_ylabel('Pool temp (°F)')
ax.set_title('Waco BSR — Bias Correction Comparison')
ax.legend(fontsize=9, loc='upper left', ncol=3)
ax.grid(True, alpha=0.25)

# — Middle: delta from baseline (negative = more cooling)
ax2 = axes[1]
ax2.plot(dates_s, mult_F  - none_F, color='#2196f3', lw=1.8, label='evap_mult − baseline')
ax2.plot(dates_s, floor_F - none_F, color='#43a047', lw=1.6, ls='-.', label='wind_floor − baseline')
ax2.plot(dates_s, pen_F   - none_F, color='#ab47bc', lw=1.6, ls=':', label='penman − baseline')
ax2.axhline(0, color='black', lw=0.8)
ax2.fill_between(dates_s, mult_F-none_F, 0,
                 where=mult_F-none_F<0, color='#2196f3', alpha=0.15, label='_')
ax2.set_ylabel('ΔT vs baseline (°F)')
ax2.set_title('Temperature change relative to uncorrected baseline (negative = cooler)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.25)

# — Bottom: bias vs 91°F cap on hot days
ax3 = axes[2]
x = np.arange(len(dates_s))
w = 0.22
ax3.bar(x-w*1.5, none_F-REAL_MAX_F,  width=w, color='#aaa',    alpha=0.85, label='none')
ax3.bar(x-w*0.5, mult_F-REAL_MAX_F,  width=w, color='#2196f3', alpha=0.85, label='evap_mult')
ax3.bar(x+w*0.5, floor_F-REAL_MAX_F, width=w, color='#43a047', alpha=0.85, label='wind_floor')
ax3.bar(x+w*1.5, pen_F-REAL_MAX_F,   width=w, color='#ab47bc', alpha=0.85, label='penman')
ax3.axhline(0, color='red', lw=1.3, ls='--', label=f'{REAL_MAX_F}°F cap')
ax3.set_xticks(x[::max(1,len(x)//15)])
ax3.set_xticklabels([dates_s[i].strftime('%b %d') for i in range(0,len(dates_s),max(1,len(dates_s)//15))],
                    rotation=30, ha='right')
ax3.set_ylabel('Pool temp − 91°F (°F)')
ax3.set_title('Bias vs 91°F cap — positive = above real-world max')
ax3.legend(fontsize=9, ncol=2)
ax3.grid(True, alpha=0.25, axis='y')

plt.tight_layout()
plt.show()

## Plot 3 — All locations: max pool temp by method
Bar chart comparing the worst-case forecast across all 5 pools.

In [ ]:
methods = ['none','evap_multiplier','wind_floor','penman']
colors  = ['#aaa','#2196f3','#43a047','#ab47bc']
labels  = ['baseline','evap_mult ×1.25','wind_floor 1.5','penman']

loc_names = list(all_data.keys())
results   = {loc: {} for loc in loc_names}

for loc in loc_names:
    rows = all_data[loc]
    if not rows:
        continue
    seed = float(rows[0]['pool_C'])
    T = {m: seed for m in methods}
    pools = {m: [] for m in methods}
    for r in rows:
        air_C = float(r['air_C'])
        RH    = float(r['humidity'] or 50.0)
        wind  = float(r['wind_speed'] or 3.0)
        solar = float(r['solar_radiation']) if r['solar_radiation'] is not None else 15.0
        for m in methods:
            T[m], _ = step(T[m], air_C, RH, wind, solar, method=m)
            pools[m].append(c2f(T[m]))
    for m in methods:
        results[loc][m] = max(pools[m])

x   = np.arange(len(loc_names))
w   = 0.20
fig, ax = plt.subplots(figsize=(13, 6))

for i, (m, label, color) in enumerate(zip(methods, labels, colors)):
    vals = [results[loc][m] for loc in loc_names]
    bars = ax.bar(x + (i-1.5)*w, vals, width=w, color=color, alpha=0.85, label=label)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.0f}', ha='center', va='bottom',
                fontsize=7.5, color='black')

ax.axhline(REAL_MAX_F, color='red', ls='--', lw=1.5, label=f'{REAL_MAX_F}°F real-world cap')
ax.set_xticks(x)
short_names = [n.replace(' Virginia Beach','\nVA Beach').replace('Atlantic Park','Atlantic\nPark') for n in loc_names]
ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylabel('Max modelled pool temp (°F)')
ax.set_title('Max Pool Temperature by Location and Correction Method')
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.25, axis='y')
plt.tight_layout()
plt.show()

## Summary — pick your method
Run this cell last. Prints the decision table.

In [ ]:
rows = all_data['Waco']
seed = float(rows[0]['pool_C'])
T = {m: seed for m in methods}
pools = {m: [] for m in methods}
air_vals = []
for r in rows:
    air_C = float(r['air_C']); RH = float(r['humidity'] or 50.0)
    wind  = float(r['wind_speed'] or 3.0)
    solar = float(r['solar_radiation']) if r['solar_radiation'] is not None else 15.0
    air_vals.append(c2f(air_C))
    for m in methods:
        T[m], _ = step(T[m], air_C, RH, wind, solar, method=m)
        pools[m].append(c2f(T[m]))

air_arr = np.array(air_vals)
hot  = air_arr > 90
mild = air_arr < 75
base = np.array(pools['none'])

print(f'{"Method":<24}  {"Max":>7}  {"Days>91F":>9}  {"HotBias":>9}  {"MildΔ":>7}  {"MeanΔ":>7}')
print('─' * 70)
for m, label in zip(methods, labels):
    p = np.array(pools[m])
    hot_bias  = (p[hot]  - REAL_MAX_F).mean() if hot.sum()  else float('nan')
    mild_delta = (p[mild] - base[mild]).mean() if mild.sum() else float('nan')
    mean_delta = (p - base).mean()
    print(f'{label:<24}  {p.max():>6.1f}°F  {(p>REAL_MAX_F).sum():>9}  '
          f'{hot_bias:>+8.1f}°F  {mild_delta:>+6.1f}°F  {mean_delta:>+6.1f}°F')

print()
print('HotBias  = mean(pool - 91°F) on days where air > 90°F  (negative = under cap ✓)')
print('MildΔ    = mean(method - baseline) on mild days        (near 0 is best)')
print('MeanΔ    = average correction applied across all days')